# Sliding Window Memory

> **Keep only the last *k* turns: the most direct way to bound memory cost while staying conversationally coherent.**

In the [previous notebook](../01_conversation_buffer_memory/conversation_buffer_memory.ipynb) we saw that **Conversation Buffer Memory** stores everything. That causes input tokens to grow linearly per turn and cumulative cost to grow quadratically. For most chat applications, that's wasteful. Users rarely reference something they said 30 turns ago.

Imagine a whiteboard that only fits five sticky notes. When you add a sixth, you peel off the oldest one and toss it. **Sliding Window Memory** works the same way. It keeps a fixed-size window of the most recent messages and discards everything older. The last *k* messages are always available, but older messages are gone.

**By the end of this notebook you'll understand:**
- How to build a deque-based sliding window from scratch with the Anthropic SDK.
- The precise tradeoff between window size, recall ability, and token cost.
- How to run a small evaluation that measures recall at different window sizes.
- When sliding window memory is the right (and wrong) choice.

## Key Concepts

- **Window size *k***: The maximum number of messages retained. A larger *k* means better recall but higher token cost.
- **FIFO eviction**: FIFO stands for "first-in, first-out." When `len(messages) > k`, the oldest message drops from the front. The window "slides" forward.
- **`collections.deque`**: Python's double-ended queue with a `maxlen` parameter. It's a natural fit: appending to a full deque automatically discards the oldest item.
- **Recency bias**: By design, the agent remembers recent context and forgets older context. This is a feature, not a bug, when recent context is what matters most.
- **Message-count vs. turn-count windows**: A message window of *k* counts each message individually (user or assistant). A turn window of *k* keeps the last *k* user-assistant pairs intact. Turn-count windows avoid orphaned messages (a response without its question).
- **Constant token cost**: Unlike buffer memory, per-turn token cost is bounded by the window size. This gives you predictable latency and cost.

## Architecture

<p align="center">
  <img src="../../images/diagrams/02_sliding_window_memory.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    participant U as User
    participant W as Sliding Window (k=4 messages)
    participant L as LLM (Claude)

    Note over W: Window: []

    U->>W: "Hi, I'm Alice" (msg 1)
    W->>L: [msg1]
    L-->>W: "Hello Alice!" (msg 2)
    Note over W: Window: [msg1, msg2]

    U->>W: "I like Python" (msg 3)
    W->>L: [msg1, msg2, msg3]
    L-->>W: "Python is great!" (msg 4)
    Note over W: Window: [msg1, msg2, msg3, msg4] - FULL

    U->>W: "What's my name?" (msg 5)
    Note over W: ⚠ Window full → drop msg1
    W->>W: Evict msg1, append msg5
    W->>L: [msg2, msg3, msg4, msg5]
    L-->>W: "Your name is Alice." (msg 6)
    Note over W: Drop msg2 → Window: [msg3, msg4, msg5, msg6]

    U->>W: "What language do I like?"
    Note over W: msg3 ("I like Python") still<br/>in window → agent remembers
    W->>L: [msg4, msg5, msg6, msg7]
    L-->>W: "You said you like Python!"
    Note over W: Eventually msg3 slides out<br/>and that fact is forgotten
```

</details>

In [ ]:
# Install required packages (run once)
%pip install -q anthropic python-dotenv matplotlib pandas

## Setup

Import the Anthropic SDK and standard helpers. We also bring in `deque` from Python's `collections` module. A **deque** ("deck") is a double-ended queue that can hold a fixed number of items. When you add one more, the oldest item drops off automatically.

In [ ]:
import os
import copy
from collections import deque
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from .env

import anthropic

assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your .env file"
print("✓ Anthropic API key loaded")

## Core Implementation

The key insight: Python's `collections.deque(maxlen=k)` gives us sliding-window behavior for free. When you `append()` to a full deque, the oldest item on the opposite end drops automatically.

In [ ]:
class SlidingWindowMemory:
    """Sliding window memory that keeps the last *k* messages."""

    def __init__(
        self,
        window_size: int = 10,
        model: str = "claude-sonnet-4-20250514",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.client = anthropic.Anthropic()
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens

        # The sliding window - a deque with a fixed max length
        self.window_size = window_size
        self.messages: deque[dict] = deque(maxlen=window_size)

        # Track all messages for analysis (not sent to the LLM)
        self.full_history: list[dict] = []
        self.turn_token_usage: list[dict] = []



Next we add the `chat` method and inspection helpers. The `chat` method converts the deque to a plain list before sending it to the API. It also records token usage and the current window size for each turn. The inspection helpers let you peek at what the LLM sees (the window) versus the full history.

In [ ]:
    # ── Chat ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        """Send a message; only the last *k* messages are sent to the LLM."""
        user_msg = {"role": "user", "content": user_input}
        self.messages.append(user_msg)
        self.full_history.append(user_msg)

        kwargs = dict(
            model=self.model,
            max_tokens=self.max_tokens,
            messages=list(self.messages),  # deque → list for the API
        )
        if self.system_prompt:
            kwargs["system"] = self.system_prompt

        response = self.client.messages.create(**kwargs)
        assistant_text = response.content[0].text

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.messages.append(assistant_msg)
        self.full_history.append(assistant_msg)

        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "window_msgs": len(self.messages),
        })

        return assistant_text

    # ── Inspection helpers ───────────────────────────────────────
    def get_window(self) -> list[dict]:
        """Return the current window contents (what the LLM sees)."""
        return list(self.messages)

    def get_full_history(self) -> list[dict]:
        """Return all messages ever sent, including evicted ones."""
        return copy.deepcopy(self.full_history)

    @property
    def evicted_count(self) -> int:
        """Number of messages that have been dropped from the window."""
        return len(self.full_history) - len(self.messages)

    def clear(self) -> None:
        self.messages.clear()
        self.full_history.clear()
        self.turn_token_usage.clear()

    def __repr__(self) -> str:
        return (
            f"SlidingWindowMemory(window={len(self.messages)}/{self.window_size}, "
            f"total={len(self.full_history)} messages, "
            f"evicted={self.evicted_count})"
        )

print("✓ SlidingWindowMemory class defined")

## Usage Example: Watching the Window Slide

Let's use a small window (`k=6` messages, which is 3 turns) and watch what happens as the conversation grows. We'll plant a fact early and then ask about it after it falls out of the window.

In [ ]:
mem = SlidingWindowMemory(
    window_size=6,  # 3 full turns (user + assistant each)
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
)

conversation = [
    "My name is Alice and I'm a pilot.",           # turn 1 - plants a fact
    "I fly Boeing 737s for a regional airline.",    # turn 2 - plants another fact
    "What's the weather like in Seattle today?",   # turn 3 - unrelated filler
    "What's my name and what do I do for work?",   # turn 4 - recall test (turn 1 may be evicted)
]

for msg in conversation:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print(f"   📊 Window: {len(mem.messages)}/{mem.window_size} msgs | Evicted: {mem.evicted_count}")
    print()

Let's inspect what the LLM currently sees versus what it has forgotten. Messages inside the window are visible to the model. Evicted messages are gone forever.

In [ ]:
# Inspect what the LLM currently sees vs. what it has forgotten
print("=== Messages IN the window (LLM can see) ===")
for i, msg in enumerate(mem.get_window()):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

print(f"\n=== Evicted messages (LLM cannot see): {mem.evicted_count} ===" )
evicted = mem.get_full_history()[:mem.evicted_count]
for i, msg in enumerate(evicted):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

## Eval: Recency vs. Recall at Different Window Sizes

Here's the key question: **how does window size affect the agent's ability to recall facts planted at different points in the conversation?**

We'll run a controlled experiment:
1. Plant 5 distinct facts at turns 1-5.
2. Add filler turns to push older facts out of smaller windows.
3. Ask the model to recall each fact.
4. Score recall accuracy at window sizes k = 4, 8, 12, and 20 messages.

In [ ]:
import re

FACTS = [
    ("My name is Elena.", "elena"),
    ("I was born in Prague.", "prague"),
    ("My favorite color is teal.", "teal"),
    ("I have two cats named Salt and Pepper.", "salt"),
    ("I work as a data scientist at a biotech company.", "data scientist"),
]

FILLER_MESSAGES = [
    "Tell me a fun fact about octopuses.",
    "What's the capital of Mongolia?",
    "How does photosynthesis work in one sentence?",
    "What year was the Eiffel Tower built?",
    "Name a famous mathematician.",
]

RECALL_QUESTIONS = [
    ("What is my name?", "elena"),
    ("Where was I born?", "prague"),
    ("What is my favorite color?", "teal"),
    ("What are my cats' names?", "salt"),
    ("What do I do for work?", "data scientist"),
]




Now we define the evaluation function. It plants facts, adds filler turns to push old facts out of small windows, then asks recall questions. Each answer is checked for a keyword to determine whether the agent remembered the fact.

In [ ]:
def run_recall_eval(window_size: int) -> dict:
    """Run the recall eval for a given window size and return scores."""
    mem = SlidingWindowMemory(
        window_size=window_size,
        system_prompt="You are a helpful assistant. Answer concisely.",
    )

    # Phase 1: Plant facts (5 turns = 10 messages)
    for fact, _ in FACTS:
        mem.chat(fact)

    # Phase 2: Filler turns to push old facts out (5 turns = 10 messages)
    for filler in FILLER_MESSAGES:
        mem.chat(filler)

    # Phase 3: Ask recall questions and check answers
    results = {}
    for question, keyword in RECALL_QUESTIONS:
        answer = mem.chat(question)
        recalled = keyword.lower() in answer.lower()
        results[keyword] = {
            "question": question,
            "answer": answer[:100],
            "recalled": recalled,
        }

    score = sum(1 for r in results.values() if r["recalled"])
    return {
        "window_size": window_size,
        "score": score,
        "total": len(RECALL_QUESTIONS),
        "pct": score / len(RECALL_QUESTIONS) * 100,
        "details": results,
        "total_input_tokens": sum(t["input_tokens"] for t in mem.turn_token_usage),
    }




Run the evaluation across four window sizes: 4, 8, 12, and 20 messages. Smaller windows will forget facts planted early in the conversation. Larger windows cost more tokens but recall more.

In [ ]:
# Test multiple window sizes
WINDOW_SIZES = [4, 8, 12, 20]
eval_results = []

for k in WINDOW_SIZES:
    print(f"Running eval with window_size={k}...", end=" ")
    result = run_recall_eval(k)
    eval_results.append(result)
    print(f"Recall: {result['score']}/{result['total']} ({result['pct']:.0f}%)")

print("\nDone!")

Plot the results. The left chart shows recall accuracy at each window size. The right chart shows total token cost. You'll see the core tradeoff: larger windows remember more but spend more tokens.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

window_sizes = [r["window_size"] for r in eval_results]
scores = [r["pct"] for r in eval_results]
tokens = [r["total_input_tokens"] for r in eval_results]

# Left: Recall accuracy vs window size
colors = ["#ef4444" if s < 40 else "#f59e0b" if s < 80 else "#22c55e" for s in scores]
bars = ax1.bar(range(len(window_sizes)), scores, color=colors, alpha=0.85, width=0.6)
ax1.set_xticks(range(len(window_sizes)))
ax1.set_xticklabels([f"k={k}" for k in window_sizes])
ax1.set_ylabel("Recall Accuracy (%)")
ax1.set_title("Recall Accuracy vs. Window Size")
ax1.set_ylim(0, 110)
ax1.axhline(y=100, color="gray", linestyle="--", alpha=0.3)
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f"{score:.0f}%", ha="center", va="bottom", fontweight="bold")

# Right: Total token cost vs window size
ax2.plot(window_sizes, tokens, "o-", color="#6366f1", linewidth=2, markersize=8)
ax2.fill_between(window_sizes, tokens, alpha=0.1, color="#6366f1")
ax2.set_xlabel("Window Size (k messages)")
ax2.set_ylabel("Total Input Tokens")
ax2.set_title("Total Token Cost vs. Window Size")
for x, y in zip(window_sizes, tokens):
    ax2.annotate(f"{y:,}", (x, y), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("sliding_window_eval.png", dpi=150, bbox_inches="tight")
plt.show()

print("The tradeoff is clear: larger windows recall more but cost more tokens.")

Build a recall matrix showing which specific facts survived at each window size. This table makes it clear that older facts (planted first) are the first to be forgotten as the window shrinks.

In [ ]:
import pandas as pd

# Build a summary table
rows = []
for r in eval_results:
    for keyword, detail in r["details"].items():
        rows.append({
            "Window Size": r["window_size"],
            "Fact": keyword,
            "Recalled?": "✓" if detail["recalled"] else "✗",
            "Answer (preview)": detail["answer"][:60],
        })

df = pd.DataFrame(rows)
# Pivot so facts are columns and window sizes are rows
pivot = df.pivot(index="Window Size", columns="Fact", values="Recalled?")
print("Recall matrix (✓ = remembered, ✗ = forgotten):\n")
print(pivot.to_string())

## Token Cost: Sliding Window vs. Full Buffer

The whole point of a sliding window is **bounded, predictable cost**. Let's compare the token growth curves side-by-side.

In [ ]:
# Simulate token growth for buffer vs sliding window
# Using a simple model: each message ≈ 50 tokens, system prompt ≈ 30 tokens
MSG_TOKENS = 50
SYS_TOKENS = 30
NUM_TURNS = 25
WINDOW_K = 10  # 5 turns

buffer_input = []
window_input = []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2  # user + assistant per turn
    buffer_input.append(SYS_TOKENS + n_msgs * MSG_TOKENS)
    window_msgs = min(n_msgs, WINDOW_K)
    window_input.append(SYS_TOKENS + window_msgs * MSG_TOKENS)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, buffer_input, "o-", color="#ef4444", label="Full Buffer", linewidth=2)
ax.plot(turns, window_input, "s-", color="#22c55e", label=f"Sliding Window (k={WINDOW_K})", linewidth=2)
ax.fill_between(turns, window_input, buffer_input, alpha=0.08, color="#ef4444")
ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per API Call")
ax.set_title("Input Tokens per Turn: Full Buffer vs. Sliding Window")
ax.legend()
ax.grid(True, alpha=0.2)

# Annotate the savings at the last turn
savings = buffer_input[-1] - window_input[-1]
ax.annotate(
    f"Savings: {savings:,} tokens/turn",
    xy=(NUM_TURNS, (buffer_input[-1] + window_input[-1]) / 2),
    fontsize=11, fontweight="bold", color="#6366f1",
    ha="right",
)

plt.tight_layout()
plt.savefig("buffer_vs_window.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn {NUM_TURNS}:")
print(f"  Full buffer:    {buffer_input[-1]:,} input tokens")
print(f"  Sliding window: {window_input[-1]:,} input tokens")
print(f"  Savings:        {savings:,} tokens ({savings/buffer_input[-1]*100:.0f}%)")

## Discussion and Tradeoffs

### Strengths
- **Predictable cost**: Token usage per turn is bounded by the window size, regardless of conversation length.
- **Quick to build**: Python's `deque(maxlen=k)` gives you the whole mechanism in one line.
- **Good enough for most chats**: Users rarely reference something from 20+ turns ago.
- **Low latency**: Bounded input size means bounded response time.

### Weaknesses
- **Hard information cutoff**: Once a message slides out of the window, it's gone. There's no trace, no summary, nothing.
- **No graceful degradation**: Unlike summary memory, the agent doesn't even know it forgot something.
- **Orphaned messages**: A message-count window can split a user-assistant pair. That leaves a response without its corresponding question. (Our implementation avoids this by counting messages, but turn-based windows are another option.)
- **Choosing *k* is tricky**: Too small and it forgets important context. Too large and it defeats the purpose.

### Choosing the Right Window Size

| Window Size | Good For | Watch Out |
|-------------|----------|-----------|
| k = 4-6 msgs | Quick Q&A, stateless tasks | Forgets context after 2-3 turns |
| k = 10-20 msgs | Most chatbots, customer support | Costs rise; may still forget early facts |
| k = 40+ msgs | Complex multi-step tasks | Approaching buffer memory cost; consider a hybrid |

### Rule of thumb
Start with `k = 10` (5 complete turns) and adjust based on your recall requirements and budget.

### When to Use Something Else
- If you need to remember facts from much earlier: use **Summary Memory** or **Entity Memory**.
- If you need to balance recency with cost but can't afford hard cutoffs: use a **Summary + Buffer Hybrid**.
- If you need to stay under an exact token budget: use **Token Buffer Memory**.

## Further Reading

- [Anthropic Messages API: Multi-turn Conversations](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LangChain ConversationBufferWindowMemory](https://python.langchain.com/docs/modules/memory/types/buffer_window?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [LlamaIndex ChatMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/chat_memory_buffer/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- [Lilian Weng, "LLM Powered Autonomous Agents" (Memory section)](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)
- Python `collections.deque`: [Official docs](https://docs.python.org/3/library/collections.html#collections.deque)

---

*← Previous: 01: Conversation Buffer Memory · Next: [03: Summary Memory](../03_summary_memory/) →*

In [ ]:
# Clean up temp files created during the demo
import os
for f in ["sliding_window_eval.png", "buffer_vs_window.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Eviction callback
Add an `on_evict` callback to `SlidingWindowMemory` that is called with each evicted message. Log evicted messages to a list and print them at the end. This gives you visibility into what the agent forgets.

### Challenge 2: Window size vs. recall heatmap
Run `run_recall_eval()` with window sizes [2, 4, 6, 8, 10, 16, 20] and plot a heatmap of recall scores by window size and question category. Identify the smallest window that achieves at least 80% recall on identity questions.

### Challenge 3: Sliding window with summary fallback
When a message is evicted from the deque, feed it to an LLM to update a running summary (like 03 Summary Memory). Prepend that summary to the context on each turn. Measure whether this hybrid beats a pure sliding window on the recall benchmark.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--02-sliding-window-memory--sliding-window-memory)
